In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Ensure the notebook can find the /src directory
root_path = Path.cwd().parent
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

In [2]:
from src.data_loader import VesselDataLoader
from src.visualizer import VesselVisualizer
from src.data_processing import engineer_telemetry_features
from src.mission_profiler import MissionProfiler

In [12]:
filename1 = "Rotherhithe_voy_179.csv"
filename2 = "Wembley_voy_236.csv"

filename = filename1 

raw_data_file = root_path / "data" / "raw" / filename

loader = VesselDataLoader(raw_data_file)
raw_df = loader.load_and_clean()

processed_df = engineer_telemetry_features(raw_df, filter_method='savgol')

profiler = MissionProfiler()

phased_df = profiler.classify_phases(processed_df)
global_stats = profiler.extract_global_statistics(phased_df)

registry_df = profiler.generate_phase_registry(phased_df, source_file_name=filename, include_loitering=False)
display(registry_df.head())

,Source_File,Start_Time,Duration_h,Energy_kWh,Mean_Power_kW,H2_Rate_Lower_kg_h,H2_Rate_Upper_kg_h,Fatigue_Damage_Rate,Mean_Power_Fluctuation_Intensity,Stay_ID,PHASE,Loitering_Handling
0,Rotherhithe_voy_179.csv,2025-12-22 10:50,27.750000,11852.221476,427.107080,23.306072,28.485199,2568.856570,2.758114,2,Sea_Transit_Ballast,Without_Loitering
1,Rotherhithe_voy_179.csv,2025-12-23 15:00,17.083333,6308.025577,369.250278,20.148984,24.626536,7887.128611,7.869740,3,Port_Idle,Included
2,Rotherhithe_voy_179.csv,2025-12-25 18:30,0.250000,105.067695,420.270781,22.933034,28.029264,45.726420,0.321269,5,Port_Idle,Included
3,Rotherhithe_voy_179.csv,2025-12-26 03:50,0.083333,35.956876,431.482515,23.544828,28.777012,0.000000,2.900009,7,Port_Idle,Included
4,Rotherhithe_voy_179.csv,2025-12-26 04:40,0.750000,327.086506,436.115342,23.797629,29.085991,480.617371,1.344436,9,Port_Idle,Included


In [13]:
plotter = VesselVisualizer(phased_df)
fig_bricks = plotter.plot_brick_space(registry_df,
                                      y_axis_metric='Mean_Power_Fluctuation_Intensity'
                                      )
fig_bricks.show()
fig_stats = plotter.plot_phase_statistics(global_stats)
fig_stats.show()